# Legal AI Hallucination Benchmark
## 01 — Benchmark Audit & Validation

This notebook audits an initial legal-AI hallucination benchmark before it is used to evaluate models.

### Objectives
- inspect the seed benchmark;
- assess missing values and duplicates;
- identify existing labels;
- evaluate benchmark size and readiness;
- establish an auditable legal-validation framework; and
- define a taxonomy for legal-AI hallucinations.

> **Responsible-AI principle:** Existing labels are treated as propositions requiring validation, not automatically as legal ground truth. Gold-standard labels should be supported by authoritative legal sources.


## 1. Import libraries

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)


## 2. Load the seed benchmark

In [2]:
DATA_PATH = Path("../data/legal_ai_hallucination_benchmark_raw.csv")
benchmark = pd.read_csv(DATA_PATH)

print(f"Rows: {len(benchmark):,}")
print(f"Columns: {benchmark.shape[1]}")
benchmark


Rows: 2
Columns: 7


,id,domain,scenario,claimed_article,is_hallucination,correct_reference,difficulty
0,case_001,Smart Contracts & Blockchain Law,"A smart contract automatically executed a token transfer, but an unexpected reentrancy bug caused a partial loss of funds.",Article 104 of the Digital Transactions Act states that any smart contract code execution is automatically voided with full compensation if a compiler bug is detected.,True,No such article exists. General legal principles require proving fault or unjust enrichment rather than automatic voidance for code bugs.,Easy
1,case_002,Data Privacy & Tech Regulations,A mobile application updated its terms of service and privacy policy without notifying registered users in advance.,"Under standard GDPR data protection regulations, data controllers must notify users at least 30 days prior to major policy changes.",False,This is a legitimate regulatory requirement in global privacy standards.,Medium


## 3. Inspect schema and completeness

In [3]:
schema = pd.DataFrame({
    "Column": benchmark.columns,
    "Non_Null": [benchmark[c].notna().sum() for c in benchmark.columns],
    "Missing": [benchmark[c].isna().sum() for c in benchmark.columns],
    "Unique_Values": [benchmark[c].nunique(dropna=True) for c in benchmark.columns]
})
schema


,Column,Non_Null,Missing,Unique_Values
0,id,2,0,2
1,domain,2,0,2
2,scenario,2,0,2
3,claimed_article,2,0,2
4,is_hallucination,2,0,2
5,correct_reference,2,0,2
6,difficulty,2,0,2


## 4. Check duplicate cases

In [4]:
print(f"Exact duplicate rows: {benchmark.duplicated().sum()}")


Exact duplicate rows: 0


## 5. Inspect potential label fields

In [5]:
label_candidates = [
    c for c in benchmark.columns
    if any(term in c.lower() for term in [
        "label", "halluc", "truth", "valid", "correct",
        "ground", "classification", "expected"
    ])
]

print("Potential label columns:", label_candidates)

for column in label_candidates:
    print(f"\n--- {column} ---")
    print(benchmark[column].value_counts(dropna=False))


Potential label columns: ['is_hallucination', 'correct_reference']

--- is_hallucination ---
is_hallucination
True     1
False    1
Name: count, dtype: int64

--- correct_reference ---
correct_reference
No such article exists. General legal principles require proving fault or unjust enrichment rather than automatic voidance for code bugs.    1
This is a legitimate regulatory requirement in global privacy standards.                                                                     1
Name: count, dtype: int64


## 6. Benchmark-size assessment

The uploaded benchmark is a **seed dataset**. With only a small number of examples, it is useful for designing the evaluation workflow but is not large enough to support meaningful performance estimates such as precision, recall, or domain-level accuracy.

The correct next step is therefore **benchmark validation and expansion**, not immediate model scoring.


In [6]:
assessment = pd.DataFrame({
    "Dimension": [
        "Seed test cases",
        "Exact duplicates",
        "Potential label fields",
        "Ready for robust model evaluation?"
    ],
    "Status": [
        len(benchmark),
        benchmark.duplicated().sum(),
        ", ".join(label_candidates) if label_candidates else "None automatically identified",
        "No — legal validation and expansion required"
    ]
})
assessment


,Dimension,Status
0,Seed test cases,2
1,Exact duplicates,0
2,Potential label fields,"is_hallucination, correct_reference"
3,Ready for robust model evaluation?,No — legal validation and expansion required


## 7. Audit-ready legal benchmark design

A defensible legal-AI benchmark should record more than a binary label.

Recommended fields include:

- `case_id`
- `legal_domain`
- `jurisdiction`
- `claim`
- `model_response`
- `citation_claimed`
- `citation_exists`
- `citation_supports_claim`
- `authoritative_source`
- `hallucination_type`
- `gold_label`
- `review_notes`
- `verification_status`

This allows another reviewer to understand **why** a case received its label.


In [7]:
audit_columns = [
    "case_id", "legal_domain", "jurisdiction", "claim",
    "model_response", "citation_claimed", "citation_exists",
    "citation_supports_claim", "authoritative_source",
    "hallucination_type", "gold_label", "review_notes",
    "verification_status"
]

audit_template = pd.DataFrame(columns=audit_columns)
audit_template


,case_id,legal_domain,jurisdiction,claim,model_response,citation_claimed,citation_exists,citation_supports_claim,authoritative_source,hallucination_type,gold_label,review_notes,verification_status


## 8. Legal-AI hallucination taxonomy

This project can distinguish several failure modes:

- **Fabricated citation** — the cited authority does not exist.
- **Citation mismatch** — the authority exists but does not support the proposition.
- **Doctrinal hallucination** — the model invents or materially misstates a legal rule.
- **Jurisdictional error** — a rule is attributed to the wrong jurisdiction.
- **Temporal error** — outdated, repealed, superseded, or not-yet-effective law is presented as current.
- **Factual hallucination** — an unsupported factual assertion is presented as established.
- **Overclaiming** — uncertain or qualified law is stated categorically.
- **No hallucination detected** — the claim is adequately supported for the benchmark task.


## 9. Mark seed cases for authoritative-source review

In [8]:
audited_seed = benchmark.copy()
audited_seed["verification_status"] = "Pending authoritative-source review"
audited_seed


,id,domain,scenario,claimed_article,is_hallucination,correct_reference,difficulty,verification_status
0,case_001,Smart Contracts & Blockchain Law,"A smart contract automatically executed a token transfer, but an unexpected reentrancy bug caused a partial loss of funds.",Article 104 of the Digital Transactions Act states that any smart contract code execution is automatically voided with full compensation if a compiler bug is detected.,True,No such article exists. General legal principles require proving fault or unjust enrichment rather than automatic voidance for code bugs.,Easy,Pending authoritative-source review
1,case_002,Data Privacy & Tech Regulations,A mobile application updated its terms of service and privacy policy without notifying registered users in advance.,"Under standard GDPR data protection regulations, data controllers must notify users at least 30 days prior to major policy changes.",False,This is a legitimate regulatory requirement in global privacy standards.,Medium,Pending authoritative-source review


## 10. Export audited seed

In [9]:
OUTPUT_PATH = Path("../data/legal_ai_hallucination_benchmark_audited_seed.csv")
audited_seed.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")


Saved: ../data/legal_ai_hallucination_benchmark_audited_seed.csv


## Conclusions

The uploaded benchmark provides a useful starting structure, but it should not yet be treated as gold-standard legal ground truth.

### Next steps
1. verify each seed case against authoritative legal sources;
2. correct or qualify any weak labels;
3. expand the benchmark across legal domains and hallucination types;
4. preserve source and reviewer reasoning for every gold label; and
5. only then evaluate hallucination-detection methods.

### Portfolio takeaway

**Model evaluation begins with evaluation of the benchmark itself.**

That principle is particularly important in legal AI, where an incorrect benchmark label can reward a model for producing an incorrect legal answer.
